# 임베딩 유사도 계산 과정 확인

이 노트북은 `비슷한 안전 대안` API가 사용하는 현재 로직을 같은 데이터로 따라갑니다.

- 사전 생성된 `snack_embeddings.npy`, `snack_embeddings_ids.npy`를 로드합니다.
- 상품명 + 원재료명으로 만든 임베딩을 사용합니다.
- 안전 조건과 등급 조건으로 후보를 거른 뒤, 코사인 유사도 상위 `top_k`개를 보여 줍니다.

실행은 `backend` 디렉터리에서 Jupyter를 시작하는 경우가 가장 간단합니다.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display


def locate_backend_dir() -> Path:
    current = Path.cwd().resolve()
    for root in [current, *current.parents]:
        for candidate in (root, root / "backend"):
            marker = candidate / "app" / "services" / "similarity_service.py"
            if marker.exists():
                return candidate
    raise FileNotFoundError("backend/app/services/similarity_service.py not found")


BACKEND_DIR = locate_backend_dir()
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from app.services.filter_engine import (  # noqa: E402
    compute_nutrition_score,
    compute_safe_snack_score,
    filter_safe_products,
)
from app.services.product_filters import normalize_conditions  # noqa: E402
from app.services.product_repository import get_base_df  # noqa: E402
from app.services.product_serializer import serialize_product  # noqa: E402
from app.services.similarity_repository import get_embeddings  # noqa: E402
from app.services.similarity_service import (  # noqa: E402
    _grade_rank,
    _is_better_alternative,
    find_similar,
)

print("backend:", BACKEND_DIR)

backend: C:\Users\yooju\.vscode\safesnack\snack-safe-app\backend


## 1. 임베딩 파일 로드

`similarity_repository.get_embeddings()`는 서버와 동일하게 임베딩 행렬과 `stable_id -> 행 인덱스` 매핑을 로드합니다. 임베딩은 생성 시 `normalize_embeddings=True`로 L2 정규화되었으므로 벡터의 크기가 거의 1인지 확인합니다.

In [2]:
df = get_base_df().copy()
bundle = get_embeddings()
assert bundle is not None, "Embedding files are missing. Run: python -m app.scripts.build_embeddings"

matrix, id_to_index = bundle
norms = np.linalg.norm(matrix, axis=1)

display(pd.DataFrame({
    "productCount": [len(df)],
    "embeddingRows": [matrix.shape[0]],
    "embeddingDimensions": [matrix.shape[1]],
    "minVectorNorm": [norms.min()],
    "maxVectorNorm": [norms.max()],
}))
print("all product ids mapped:", df["stable_id"].astype(str).isin(id_to_index).all())

,productCount,embeddingRows,embeddingDimensions,minVectorNorm,maxVectorNorm
0,539,539,384,1.0,1.0


all product ids mapped: True


## 2. 기준 상품과 필터 선택

아래 값을 바꿔 여러 상품의 결과를 비교할 수 있습니다. `CONDITIONS`는 안전 필터, `SELECTED_TASTES`는 `safe_snack_score` 계산에 반영되는 취향 값입니다.

In [3]:
# Change these inputs and run the cells below again.
PRODUCT_ID = "0"
CONDITIONS = []          # Example: ["\uc54c\ub808\ub974\uae30_\ub545\ucf69"]
SELECTED_TASTES = []     # Example: ["\ub2ec\ub2ec"]
TOP_K = 5

assert PRODUCT_ID in id_to_index, f"No embedding for product id: {PRODUCT_ID}"
raw_query_row = df[df["stable_id"].astype(str) == PRODUCT_ID].iloc[0]
query_product = serialize_product(raw_query_row)
source_text = query_product["name"]
if query_product["ingredientsRaw"]:
    source_text += " || " + query_product["ingredientsRaw"]

display(pd.DataFrame([{
    "id": query_product["id"],
    "brand": query_product["brand"],
    "name": query_product["name"],
    "embeddingInputPreview": source_text[:180],
}]))
print("conditions:", CONDITIONS)
print("selected tastes:", SELECTED_TASTES)

,id,brand,name,embeddingInputPreview
0,0,(주)오리온 제2익산공장,후레쉬베리 요거트베리,"후레쉬베리 요거트베리 || 밀가루, 전란액, 백설탕, 쇼트닝, 잼, 옥수수기름(옥배..."


conditions: []
selected tastes: []


## 3. 안전 점수와 등급 조건

서비스는 유사도를 계산하기 전에 `safe_snack_score`를 계산합니다. 현재 상품이 A등급이면 A 후보만, B/C/D등급이면 더 높은 등급의 후보만 유사도 계산에 진입합니다.

In [4]:
GRADE_LABELS = {0: "A", 1: "B", 2: "C", 3: "D"}

scored_df = compute_nutrition_score(df)
scored_df = compute_safe_snack_score(
    scored_df,
    selected_tastes=SELECTED_TASTES or None,
)
query_row = scored_df[scored_df["stable_id"].astype(str) == PRODUCT_ID].iloc[0]
query_grade_rank = _grade_rank(float(query_row["safe_snack_score"]))

display(pd.DataFrame([{
    "id": PRODUCT_ID,
    "name": query_product["name"],
    "nutritionScore": int(query_row["nutrition_score"]),
    "safeSnackScore": int(query_row["safe_snack_score"]),
    "grade": GRADE_LABELS[query_grade_rank],
}]))

,id,name,nutritionScore,safeSnackScore,grade
0,0,후레쉬베리 요거트베리,69,78,B


## 4. 유사도 계산 전 후보 축소 과정

후보는 다음 순서로 줄어듭니다.

1. 사용자의 질환/알레르기 조건을 통과한 상품만 남김
2. 현재 상품 자신을 제외
3. 임베딩이 있는 상품만 남김
4. 현재 등급 규칙을 통과한 상품만 남김

In [5]:
normalized_conditions = normalize_conditions(CONDITIONS)
safe_df = (
    filter_safe_products(scored_df, normalized_conditions)
    if normalized_conditions
    else scored_df.copy()
)
without_query_df = safe_df[safe_df["stable_id"].astype(str) != PRODUCT_ID].copy()
embedded_df = without_query_df[
    without_query_df["stable_id"].astype(str).isin(id_to_index)
].copy()
embedded_df["gradeRank"] = embedded_df["safe_snack_score"].apply(lambda value: _grade_rank(float(value)))
eligible_df = embedded_df[
    embedded_df["gradeRank"].apply(
        lambda rank: _is_better_alternative(query_grade_rank, int(rank))
    )
].copy()

display(pd.DataFrame([
    {"stage": "all products", "count": len(scored_df)},
    {"stage": "safe condition passed", "count": len(safe_df)},
    {"stage": "selected product removed", "count": len(without_query_df)},
    {"stage": "embedding available", "count": len(embedded_df)},
    {"stage": "grade eligible", "count": len(eligible_df)},
]))
print("query grade:", GRADE_LABELS[query_grade_rank])
print("candidate grades:", sorted({GRADE_LABELS[int(rank)] for rank in eligible_df["gradeRank"].tolist()}))

,stage,count
0,all products,539
1,safe condition passed,539
2,selected product removed,538
3,embedding available,538
4,grade eligible,219


query grade: B
candidate grades: ['A']


## 5. Dot product = cosine similarity 확인

두 벡터가 모두 L2 정규화되어 길이가 1이면, 내적(`dot product`)은 코사인 유사도와 같습니다.

```text
cosine(a, b) = (a · b) / (||a|| ||b||) = a · b
```

In [6]:
query_index = id_to_index[PRODUCT_ID]
query_vector = matrix[query_index]
candidate_ids = eligible_df["stable_id"].astype(str).tolist()
candidate_indices = [id_to_index[sid] for sid in candidate_ids]
candidate_vectors = matrix[candidate_indices]

dot_scores = candidate_vectors @ query_vector
cosine_scores = dot_scores / (
    np.linalg.norm(candidate_vectors, axis=1) * np.linalg.norm(query_vector)
)

print("query vector shape:", query_vector.shape)
print("first 10 dimensions:", query_vector[:10])
print("candidate matrix shape:", candidate_vectors.shape)
print("maximum |dot - cosine|:", float(np.max(np.abs(dot_scores - cosine_scores))) if len(dot_scores) else 0.0)

query vector shape: (384,)
first 10 dimensions: [-0.0366185  -0.02482486  0.05801939  0.02927499  0.10922629  0.01818709
  0.11987418 -0.01805558 -0.08959229 -0.02021324]
candidate matrix shape: (219, 384)
maximum |dot - cosine|: 1.1920928955078125e-07


## 6. 유사도 상위 후보 확인

최종 순위는 `safe_snack_score`의 높고 낮음이 아니라, 위 필터를 통과한 후보 중 **임베딩 코사인 유사도**가 높은 순서입니다.

In [7]:
ranked_df = eligible_df.copy()
ranked_df["similarityScore"] = dot_scores
ranked_df = ranked_df.sort_values("similarityScore", ascending=False)

top_rows = []
for _, row in ranked_df.head(TOP_K).iterrows():
    item = serialize_product(row)
    top_rows.append({
        "id": item["id"],
        "brand": item["brand"],
        "name": item["name"],
        "safeSnackScore": item["safeSnackScore"],
        "grade": GRADE_LABELS[_grade_rank(float(item["safeSnackScore"]))],
        "similarityScore": round(float(row["similarityScore"]), 4),
    })

display(pd.DataFrame(top_rows))

,id,brand,name,safeSnackScore,grade,similarityScore
0,120,(주)코스모스제과,트위스트,86,A,0.9108
1,552,(주)오리온,오징어땅콩,91,A,0.8850
2,336,(주) 청우식품,파파크래커,90,A,0.8829
3,545,영진식품,땅콩강정,94,A,0.8789
4,542,해태제과식품(주),맛동산,89,A,0.8765


## 7. 실제 서비스 결과와 대조

노트북에서 직접 계산한 상위 ID가 `find_similar()`의 응답과 일치하는지 확인합니다.

In [8]:
service_results = find_similar(
    PRODUCT_ID,
    conditions=CONDITIONS,
    selected_tastes=SELECTED_TASTES or None,
    top_k=TOP_K,
)
service_table = pd.DataFrame([
    {
        "id": item["id"],
        "name": item["name"],
        "safeSnackScore": item["safeSnackScore"],
        "similarityScore": item["similarityScore"],
    }
    for item in service_results
])
display(service_table)

notebook_ids = [row["id"] for row in top_rows]
service_ids = [item["id"] for item in service_results]
assert notebook_ids == service_ids, (notebook_ids, service_ids)
print("Matched service output ids:", service_ids)

,id,name,safeSnackScore,similarityScore
0,120,트위스트,86,0.9108
1,552,오징어땅콩,91,0.8850
2,336,파파크래커,90,0.8829
3,545,땅콩강정,94,0.8789
4,542,맛동산,89,0.8765


Matched service output ids: ['120', '552', '336', '545', '542']


## 해석 포인트

- `similarityScore`는 상품명과 원재료 텍스트의 의미적 유사성을 나타냅니다.
- `safeSnackScore`는 후보가 추천 가능한 등급인지 판정하는 데 쓰이지만, 유사 후보 사이의 마지막 순위를 직접 정하지는 않습니다.
- `CONDITIONS`를 추가하면 안전하지 않은 상품은 유사도 계산 전에 제외됩니다.
- `SELECTED_TASTES`를 추가하면 종합 점수와 등급 통과 여부가 달라져 후보 집합이 변할 수 있습니다.